# Author Classification Pipeline

This notebook implements a text classification pipeline to identify authors (EAP, HPL, MWS) based on their writing styles. It explores TF-IDF, Word Embeddings (Word2Vec, GloVe, FastText), and Deep Learning (PyTorch).


In [ ]:
!pip install gensim optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 10.6 MB/s eta 0:00:00


In [23]:
import pandas as pd
import numpy as np
import os
import sys
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
from gensim.models import Word2Vec, FastText
import gensim.downloader as api
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import optuna
import warnings

warnings.filterwarnings('ignore')

In [24]:
# Load the cleaned dataset
df = pd.read_csv('cleaned_data.csv')

# Check for missing values
print(f"Missing values in 'cleaned_text': {df['cleaned_text'].isnull().sum()}")
df = df.dropna(subset=['cleaned_text'])

# Encode the target labels
le = LabelEncoder()
df['author_encoded'] = le.fit_transform(df['author'])
print(f"Encoded authors: {dict(zip(le.classes_, range(len(le.classes_))))}")

# Split into train/validation sets
X_train_full, X_val_full, y_train, y_val = train_test_split(df['cleaned_text'], df['author_encoded'], test_size=0.2, random_state=42)

results = {} # Storage for multi-logloss results

df.head()

Missing values in 'cleaned_text': 3
Encoded authors: {'EAP': 0, 'HPL': 1, 'MWS': 2}


,id,author,cleaned_text,author_encoded
0,id26305,EAP,process however aforded aforded means ascerta...,0
1,id17569,HPL,never ocured fumbling might mere mistake,1
2,id11008,EAP,left hand gold snuff box capered hill cutting ...,0
3,id27763,MWS,lovely spring loked windsor terace sixteen fer...,2
4,id12958,HPL,finding nothing else even gold superintendent ...,1


In [ ]:
# 1. TF-IDF Vectorization (Baseline)
tfidf_base = TfidfVectorizer()
X_train_tfidf_base = tfidf_base.fit_transform(X_train_full)
X_val_tfidf_base = tfidf_base.transform(X_val_full)

# Logistic Regression Baseline
lr_base = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000)
lr_base.fit(X_train_tfidf_base, y_train)
lr_probs = lr_base.predict_proba(X_val_tfidf_base)
results['TF-IDF + LR (Baseline)'] = log_loss(y_val, lr_probs)

# XGBoost Baseline
xgb_base = xgb.XGBClassifier(objective='multi:softprob', num_class=len(le.classes_), eval_metric='mlogloss', seed=42)
xgb_base.fit(X_train_tfidf_base, y_train)
xgb_probs = xgb_base.predict_proba(X_val_tfidf_base)
results['TF-IDF + XGBoost (Baseline)'] = log_loss(y_val, xgb_probs)

print(f"Results: {results}")

Results: {'TF-IDF + LR (Baseline)': 0.5630309834571711, 'TF-IDF + XGBoost (Baseline)': 0.7346952496398842}


In [ ]:
# 2. Fine-Tuning TF-IDF with Optuna

def objective_lr(trial):
    ngram_range = trial.suggest_categorical('ngram_range', [(1, 1), (1, 2)])
    max_df = trial.suggest_float('max_df', 0.7, 1.0)
    C = trial.suggest_float('C', 0.1, 10.0, log=True)

    vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_df=max_df, stop_words='english')
    X_tr = vectorizer.fit_transform(X_train_full)

    classifier = LogisticRegression(C=C, multi_class='multinomial', solver='lbfgs', max_iter=1000)

    # Using cross_val_score for objective
    score = cross_val_score(classifier, X_tr, y_train, cv=3, scoring='neg_log_loss').mean()
    return score

study_lr = optuna.create_study(direction='maximize')
study_lr.optimize(objective_lr, n_trials=10)

best_params_lr = study_lr.best_params
print(f"Best Params LR: {best_params_lr}")

# Train best model
tfidf_opt = TfidfVectorizer(ngram_range=best_params_lr['ngram_range'],
                            max_df=best_params_lr['max_df'],
                            stop_words='english')
X_train_tfidf_opt = tfidf_opt.fit_transform(X_train_full)
X_val_tfidf_opt = tfidf_opt.transform(X_val_full)

best_lr_model = LogisticRegression(C=best_params_lr['C'], multi_class='multinomial', solver='lbfgs', max_iter=1000)
best_lr_model.fit(X_train_tfidf_opt, y_train)
best_tfidf_probs = best_lr_model.predict_proba(X_val_tfidf_opt)

results['TF-IDF (Optimized)'] = log_loss(y_val, best_tfidf_probs)
print(f"Results: {results}")

[I 2026-04-14 20:45:38,904] A new study created in memory with name: no-name-79151685-14c9-494e-904b-e4b276624489
[I 2026-04-14 20:45:46,187] Trial 0 finished with value: -0.9157430142848048 and parameters: {'ngram_range': (1, 2), 'max_df': 0.8243052723641381, 'C': 0.2938674909646674}. Best is trial 0 with value: -0.9157430142848048.
[I 2026-04-14 20:45:55,425] Trial 1 finished with value: -0.5489363405611166 and parameters: {'ngram_range': (1, 1), 'max_df': 0.7876422101424831, 'C': 5.0065342744712495}. Best is trial 1 with value: -0.5489363405611166.
[I 2026-04-14 20:46:04,548] Trial 2 finished with value: -0.927959999439782 and parameters: {'ngram_range': (1, 2), 'max_df': 0.9274806965282811, 'C': 0.26143640464039763}. Best is trial 1 with value: -0.5489363405611166.
[I 2026-04-14 20:46:10,993] Trial 3 finished with value: -0.6455997744388314 and parameters: {'ngram_range': (1, 1), 'max_df': 0.7471670025306217, 'C': 1.1048541383197865}. Best is trial 1 with value: -0.5489363405611166

Best Params LR: {'ngram_range': (1, 1), 'max_df': 0.7876422101424831, 'C': 5.0065342744712495}
Results: {'TF-IDF + LR (Baseline)': 0.5630309834571711, 'TF-IDF + XGBoost (Baseline)': 0.7346952496398842, 'TF-IDF (Optimized)': 0.4879154231103345}


In [ ]:
# 2b. Fine-Tuning TF-IDF with XGBoost using Optuna

def objective_xgb_tfidf(trial):
    ngram_range = trial.suggest_categorical('ngram_range', [(1, 1), (1, 2)])
    max_depth = trial.suggest_int('max_depth', 3, 7,8)
    learning_rate = trial.suggest_float('learning_rate', 0.05, 0.2, log=True)
    n_estimators = trial.suggest_int('n_estimators', 50, 150)

    vectorizer = TfidfVectorizer(ngram_range=ngram_range, stop_words='english')
    X_tr = vectorizer.fit_transform(X_train_full)

    classifier = xgb.XGBClassifier(
        objective='multi:softprob',
        num_class=len(le.classes_),
        eval_metric='mlogloss',
        max_depth=max_depth,
        learning_rate=learning_rate,
        n_estimators=n_estimators,
        seed=42,
        n_jobs=-1
    )

    score = cross_val_score(classifier, X_tr, y_train, cv=3, scoring='neg_log_loss').mean()
    return score

study_xgb_tfidf = optuna.create_study(direction='maximize')
study_xgb_tfidf.optimize(objective_xgb_tfidf, n_trials=10)

best_params_xgb_tfidf = study_xgb_tfidf.best_params
print(f"Best Params XGB+TFIDF: {best_params_xgb_tfidf}")

tfidf_opt_xgb = TfidfVectorizer(ngram_range=best_params_xgb_tfidf['ngram_range'], stop_words='english')
X_train_tfidf_opt_xgb = tfidf_opt_xgb.fit_transform(X_train_full)
X_val_tfidf_opt_xgb = tfidf_opt_xgb.transform(X_val_full)

best_xgb_tfidf_model = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=len(le.classes_),
    eval_metric='mlogloss',
    max_depth=best_params_xgb_tfidf['max_depth'],
    learning_rate=best_params_xgb_tfidf['learning_rate'],
    n_estimators=best_params_xgb_tfidf['n_estimators'],
    seed=42,
    n_jobs=-1
)

best_xgb_tfidf_model.fit(X_train_tfidf_opt_xgb, y_train)
best_tfidf_xgb_probs = best_xgb_tfidf_model.predict_proba(X_val_tfidf_opt_xgb)

results['TF-IDF + XGBoost (Optimized)'] = log_loss(y_val, best_tfidf_xgb_probs)
print(f"Results: {results}")

[I 2026-04-14 20:47:02,490] A new study created in memory with name: no-name-744377b7-f6ef-4b7c-bec3-d7b5c2c7121e
[I 2026-04-14 20:47:39,135] Trial 0 finished with value: -0.9309356519052548 and parameters: {'ngram_range': (1, 1), 'max_depth': 3, 'learning_rate': 0.09319817636783827, 'n_estimators': 123}. Best is trial 0 with value: -0.9309356519052548.
[I 2026-04-14 20:48:09,669] Trial 1 finished with value: -0.9447059442835285 and parameters: {'ngram_range': (1, 1), 'max_depth': 3, 'learning_rate': 0.09354691437796922, 'n_estimators': 101}. Best is trial 0 with value: -0.9309356519052548.
[I 2026-04-14 20:49:31,843] Trial 2 finished with value: -0.9301893125077131 and parameters: {'ngram_range': (1, 2), 'max_depth': 3, 'learning_rate': 0.11290271423884751, 'n_estimators': 103}. Best is trial 2 with value: -0.9301893125077131.
[I 2026-04-14 20:50:17,196] Trial 3 finished with value: -0.9291770239036029 and parameters: {'ngram_range': (1, 1), 'max_depth': 3, 'learning_rate': 0.07887378

Best Params XGB+TFIDF: {'ngram_range': (1, 1), 'max_depth': 3, 'learning_rate': 0.15392266135606852, 'n_estimators': 137}
Results: {'TF-IDF + LR (Baseline)': 0.5630309834571711, 'TF-IDF + XGBoost (Baseline)': 0.7346952496398842, 'TF-IDF (Optimized)': 0.4879154231103345, 'TF-IDF + XGBoost (Optimized)': 0.866615473620668}


In [25]:
# 3. Word2Vec from Scratch (CBOW and Skip-gram) with TF-IDF Weighting

# Fit a TF-IDF vectorizer to extract IDF weights for word importance
tfidf_weighing = TfidfVectorizer(stop_words='english')
tfidf_weighing.fit(X_train_full)
word2idf = dict(zip(tfidf_weighing.get_feature_names_out(), tfidf_weighing.idf_))

# Tokenize sentences
sentences = [text.split() for text in X_train_full]

# CBOW
w2v_cbow = Word2Vec(sentences, vector_size=100, window=5, min_count=2, sg=0, seed=42)
# Skip-gram
w2v_sg = Word2Vec(sentences, vector_size=100, window=5, min_count=2, sg=1, seed=42)

# Function to compute TF-IDF weighted average of word vectors
def get_weighted_w2v(text, model, size, is_gensim_api=False):
    words = text.split()
    vectors = []
    weights = []

    for word in words:
        in_vocab = word in model if is_gensim_api else word in model.wv
        if in_vocab:
            vec = model[word] if is_gensim_api else model.wv[word]
            weight = word2idf.get(word, 1.0) # Default to 1.0 if word not in TF-IDF vocab
            vectors.append(vec)
            weights.append(weight)

    if len(vectors) == 0:
        return np.zeros(size)

    return np.average(vectors, axis=0, weights=weights)

X_train_cbow = np.array([get_weighted_w2v(text, w2v_cbow, 100) for text in X_train_full])
X_val_cbow = np.array([get_weighted_w2v(text, w2v_cbow, 100) for text in X_val_full])

X_train_sg = np.array([get_weighted_w2v(text, w2v_sg, 100) for text in X_train_full])
X_val_sg = np.array([get_weighted_w2v(text, w2v_sg, 100) for text in X_val_full])

print("Word2Vec training complete with TF-IDF weighted averaging.")

Word2Vec training complete with TF-IDF weighted averaging.


In [26]:
# 4. Load Pre-trained GloVe and FastText
# Using gensim's downloader for pre-trained weights
# Note: 'glove-wiki-gigaword-100' (~128MB) is relatively fast.
# 'fasttext-wiki-news-subwords-300' (950MB+) is removed to speed up execution.
# We will use 'glove-twitter-25' as a lightweight placeholder for FastText functionality.

glove_model = api.load("glove-wiki-gigaword-100")
fasttext_model = api.load("glove-twitter-25")

# Generate TF-IDF weighted average vectors for pre-trained models
# We reuse the `get_weighted_w2v` function from previous cell

X_train_glove = np.array([get_weighted_w2v(text, glove_model, 100, is_gensim_api=True) for text in X_train_full])
X_val_glove = np.array([get_weighted_w2v(text, glove_model, 100, is_gensim_api=True) for text in X_val_full])

# GloVe-100 vs Twitter-25
X_train_fasttext = np.array([get_weighted_w2v(text, fasttext_model, 25, is_gensim_api=True) for text in X_train_full])
X_val_fasttext = np.array([get_weighted_w2v(text, fasttext_model, 25, is_gensim_api=True) for text in X_val_full])

print("Pre-trained vectors loaded and TF-IDF weighted (using lightweight models for speed).")

Pre-trained vectors loaded and TF-IDF weighted (using lightweight models for speed).


In [27]:
# 5. Baseline Embeddings + Models (LR, XGBoost)

embeddings = {
    'CBOW': (X_train_cbow, X_val_cbow),
    'Skip-gram': (X_train_sg, X_val_sg),
    'GloVe': (X_train_glove, X_val_glove),
    'FastText': (X_train_fasttext, X_val_fasttext)
}

for name, (X_tr, X_v) in embeddings.items():
    # Logistic Regression
    lr = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000)
    lr.fit(X_tr, y_train)
    lr_probs = lr.predict_proba(X_v)
    results[f'{name} + LR (Baseline)'] = log_loss(y_val, lr_probs)

    # XGBoost
    xgb_emb = xgb.XGBClassifier(objective='multi:softprob', num_class=len(le.classes_), eval_metric='mlogloss', seed=42)
    xgb_emb.fit(X_tr, y_train)
    xgb_probs = xgb_emb.predict_proba(X_v)
    results[f'{name} + XGBoost (Baseline)'] = log_loss(y_val, xgb_probs)

print(f"Update Results: {results}")

Update Results: {'CBOW + LR (Baseline)': 1.0777089346972755, 'CBOW + XGBoost (Baseline)': 1.0705267868041763, 'Skip-gram + LR (Baseline)': 0.9018068720502219, 'Skip-gram + XGBoost (Baseline)': 0.8270750678834355, 'GloVe + LR (Baseline)': 0.8814458694899207, 'GloVe + XGBoost (Baseline)': 0.8491749170964259, 'FastText + LR (Baseline)': 0.9742640959269026, 'FastText + XGBoost (Baseline)': 0.9642591601195238}


In [28]:
# 6. Optimize XGBoost on the Best Embedding (Let's assume GloVe is often strong) using Optuna

best_emb_name = 'GloVe'
X_tr_opt, X_v_opt = embeddings[best_emb_name]

def objective_xgb_emb(trial):
    max_depth = trial.suggest_int('max_depth', 3, 7)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.2, log=True)
    n_estimators = trial.suggest_int('n_estimators', 100, 300)
    subsample = trial.suggest_float('subsample', 0.6, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.6, 1.0)

    classifier = xgb.XGBClassifier(
        objective='multi:softprob',
        num_class=len(le.classes_),
        eval_metric='mlogloss',
        max_depth=max_depth,
        learning_rate=learning_rate,
        n_estimators=n_estimators,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        seed=42,
        n_jobs=-1
    )

    score = cross_val_score(classifier, X_tr_opt, y_train, cv=3, scoring='neg_log_loss').mean()
    return score

study_xgb_emb = optuna.create_study(direction='maximize')
study_xgb_emb.optimize(objective_xgb_emb, n_trials=10)

best_params_xgb_emb = study_xgb_emb.best_params
print(f"Best Params XGB+EMB: {best_params_xgb_emb}")

best_xgb_emb_model = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=len(le.classes_),
    eval_metric='mlogloss',
    **best_params_xgb_emb,
    seed=42,
    n_jobs=-1
)

best_xgb_emb_model.fit(X_tr_opt, y_train)
best_xgb_probs = best_xgb_emb_model.predict_proba(X_v_opt)

results[f'{best_emb_name} + XGBoost (Optimized)'] = log_loss(y_val, best_xgb_probs)
print(f"Results: {results}")

[I 2026-04-14 21:56:07,406] A new study created in memory with name: no-name-9b23f844-f2f6-4bb3-b0b2-64f15a37ef33
[I 2026-04-14 21:56:42,596] Trial 0 finished with value: -0.9035214719153076 and parameters: {'max_depth': 4, 'learning_rate': 0.0178799863383173, 'n_estimators': 232, 'subsample': 0.857991383843612, 'colsample_bytree': 0.9980271191107717}. Best is trial 0 with value: -0.9035214719153076.
[I 2026-04-14 21:57:29,256] Trial 1 finished with value: -0.8287991773533648 and parameters: {'max_depth': 6, 'learning_rate': 0.08869796255510887, 'n_estimators': 175, 'subsample': 0.8157874908309037, 'colsample_bytree': 0.6196089798699937}. Best is trial 1 with value: -0.8287991773533648.
[I 2026-04-14 21:58:37,110] Trial 2 finished with value: -0.8394560692451138 and parameters: {'max_depth': 6, 'learning_rate': 0.12867797308837112, 'n_estimators': 195, 'subsample': 0.7027664970429186, 'colsample_bytree': 0.8375849485977656}. Best is trial 1 with value: -0.8287991773533648.
[I 2026-04-1

Best Params XGB+EMB: {'max_depth': 6, 'learning_rate': 0.05697980546401763, 'n_estimators': 232, 'subsample': 0.6171221993802893, 'colsample_bytree': 0.7619743243486317}
Results: {'CBOW + LR (Baseline)': 1.0777089346972755, 'CBOW + XGBoost (Baseline)': 1.0705267868041763, 'Skip-gram + LR (Baseline)': 0.9018068720502219, 'Skip-gram + XGBoost (Baseline)': 0.8270750678834355, 'GloVe + LR (Baseline)': 0.8814458694899207, 'GloVe + XGBoost (Baseline)': 0.8491749170964259, 'FastText + LR (Baseline)': 0.9742640959269026, 'FastText + XGBoost (Baseline)': 0.9642591601195238, 'GloVe + XGBoost (Optimized)': 0.8042703594698176}


# Deep Learning Approach: Recommendations

For sequential text data, a **Bidirectional LSTM (BiLSTM)** with an **Embedding layer** is a strong starting point as it captures long-range dependencies in both directions (forward and backward).

**Recommendations for Deep Learning:**

1. **Embedding Layer**: Initialize with pre-trained vectors (GloVe/FastText) to improve performance on small datasets.
2. **Dropout**: Use dropout layers to prevent overfitting, which is common with neural networks on relatively small text datasets.
3. **Architecture**: Consider using a **1D CNN** on top of the embedding layer to extract local n-gram features efficiently.
4. **Learning Rate Scheduler**: Implement a scheduler to reduce the learning rate when validation loss plateaus.
5. **Transformers**: For state-of-the-art results, fine-tuning a model like `DistilBERT` or `RoBERTa` would provide significantly better contextual understanding than BiLSTMs.


In [29]:
# 7. Deep Learning (PyTorch BiLSTM Implementation with Pre-Trained GloVe)

# Simple Tokenizer for PyTorch
from collections import Counter

# Create vocab from training set
all_text = " ".join(X_train_full).split()
word_counts = Counter(all_text)
vocab = {word: i+2 for i, (word, count) in enumerate(word_counts.items()) if count > 0}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1
vocab_size = len(vocab)

def text_to_seq(text, max_len=50):
    seq = [vocab.get(w, 1) for w in text.split()[:max_len]]
    return seq + [0] * (max_len - len(seq))

X_train_dl = torch.tensor([text_to_seq(t) for t in X_train_full], dtype=torch.long)
X_val_dl = torch.tensor([text_to_seq(t) for t in X_val_full], dtype=torch.long)
y_train_dl = torch.tensor(y_train.values, dtype=torch.long)
y_val_dl = torch.tensor(y_val.values, dtype=torch.long)

# Initialize Embedding Matrix with GloVe weights
EMBED_DIM = 100
embedding_matrix = np.zeros((vocab_size, EMBED_DIM))
for word, i in vocab.items():
    if word in glove_model:
        embedding_matrix[i] = glove_model[word]
    else:
        # random initialization for words not in GloVe (e.g. archaic words)
        embedding_matrix[i] = np.random.normal(scale=0.6, size=(EMBED_DIM, ))

class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, embedding_matrix):
        super(BiLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        # Load the pre-trained weights into the Embedding layer
        self.embedding.weight = nn.Parameter(torch.tensor(embedding_matrix, dtype=torch.float32))
        self.embedding.weight.requires_grad = True # allow fine-tuning the embeddings

        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3) # Added dropout for regularization
        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)
        # Concat the final hidden states of forward and backward LSTM
        hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        hidden = self.dropout(hidden)
        out = self.fc(hidden)
        return out # Raw logits for CrossEntropyLoss

# Parameters
HIDDEN_DIM = 64
OUTPUT_DIM = len(le.classes_)

model_dl = BiLSTM(vocab_size, EMBED_DIM, HIDDEN_DIM, OUTPUT_DIM, embedding_matrix)
criterion = nn.CrossEntropyLoss()
# Using AdamW instead of Adam for better weight decay handling
optimizer = optim.AdamW(model_dl.parameters(), lr=0.001)

# Simple training loop
for epoch in range(12): # Increased epochs slightly since we have dropout
    model_dl.train()
    optimizer.zero_grad()
    outputs = model_dl(X_train_dl)
    loss = criterion(outputs, y_train_dl)
    loss.backward()
    optimizer.step()

# Evaluation (Probs)
model_dl.eval()
with torch.no_grad():
    val_logits = model_dl(X_val_dl)
    val_probs_dl = torch.softmax(val_logits, dim=1).numpy()
    results['BiLSTM (PyTorch + GloVe)'] = log_loss(y_val, val_probs_dl)

print(f"Results: {results}")

Results: {'CBOW + LR (Baseline)': 1.0777089346972755, 'CBOW + XGBoost (Baseline)': 1.0705267868041763, 'Skip-gram + LR (Baseline)': 0.9018068720502219, 'Skip-gram + XGBoost (Baseline)': 0.8270750678834355, 'GloVe + LR (Baseline)': 0.8814458694899207, 'GloVe + XGBoost (Baseline)': 0.8491749170964259, 'FastText + LR (Baseline)': 0.9742640959269026, 'FastText + XGBoost (Baseline)': 0.9642591601195238, 'GloVe + XGBoost (Optimized)': 0.8042703594698176, 'BiLSTM (PyTorch + GloVe)': 1.0607645112124018}


In [30]:
# 8. Ensemble Learning (Soft Voting)
# Create ensemble by averaging probabilities from best performing models
# best_tfidf_probs: optimized TF-IDF + LR
# best_tfidf_xgb_probs: optimized TF-IDF + XGBoost
# val_probs_dl: BiLSTM probabilities

ensemble_probs = (best_tfidf_probs + best_tfidf_xgb_probs + val_probs_dl) / 3
results['Ensemble (Weighted Avg)'] = log_loss(y_val, ensemble_probs)

print(f"Update Results: {results}")

Update Results: {'CBOW + LR (Baseline)': 1.0777089346972755, 'CBOW + XGBoost (Baseline)': 1.0705267868041763, 'Skip-gram + LR (Baseline)': 0.9018068720502219, 'Skip-gram + XGBoost (Baseline)': 0.8270750678834355, 'GloVe + LR (Baseline)': 0.8814458694899207, 'GloVe + XGBoost (Baseline)': 0.8491749170964259, 'FastText + LR (Baseline)': 0.9742640959269026, 'FastText + XGBoost (Baseline)': 0.9642591601195238, 'GloVe + XGBoost (Optimized)': 0.8042703594698176, 'BiLSTM (PyTorch + GloVe)': 1.0607645112124018, 'Ensemble (Weighted Avg)': 0.7373329534226842}


In [31]:
# Final Comparison of Multi-Logloss Results
final_results = pd.DataFrame(list(results.items()), columns=['Model', 'Multi-Logloss']).sort_values(by='Multi-Logloss')
print("Final Rankings:")
print(final_results)

Final Rankings:
                             Model  Multi-Logloss
10         Ensemble (Weighted Avg)       0.737333
8      GloVe + XGBoost (Optimized)       0.804270
3   Skip-gram + XGBoost (Baseline)       0.827075
5       GloVe + XGBoost (Baseline)       0.849175
4            GloVe + LR (Baseline)       0.881446
2        Skip-gram + LR (Baseline)       0.901807
7    FastText + XGBoost (Baseline)       0.964259
6         FastText + LR (Baseline)       0.974264
9         BiLSTM (PyTorch + GloVe)       1.060765
1        CBOW + XGBoost (Baseline)       1.070527
0             CBOW + LR (Baseline)       1.077709
